In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'python').exists():
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Working directory: {ROOT}')


In [1]:
# transformer model

import os, gc, json, codecs, re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, accuracy_score
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from python.preprocessing_class import Preprocessing, load_pres, load_movies

# CUDA memory frag
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Config for run
DATASET = "movies"

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
EPOCHS = 20
EARLY_STOPPING_PATIENCE = 3
GRADIENT_ACCUMULATION_STEPS = 2   # to accumulate batches

# load data
alltxt, alllabs = None, None

print("Loading DATASET", DATASET)

if DATASET == "pres": 
    alltxt, alllabs = load_pres("Dataset/corpus.tache1.learn.utf8")
    alltxt, alllabs = np.array(alltxt), np.array(alllabs)
    # Convert labels: 1 -> 0, -1 -> 1
    alllabs = np.where(alllabs == 1, 0, 1)
else:
    alltxt, alllabs = load_movies("./Dataset/movies1000/")
    alltxt, alllabs = np.array(alltxt), np.array(alllabs)
print(f"dataset size: {len(alltxt)}")

# weigths
#val, counts = np.unique(alllabs, return_counts=True)
#class_weights = 1 / counts

#print(f"weights: {class_weights}")

# splits 
X_train, X_test, y_train, y_test = train_test_split(
    alltxt, alllabs,
    test_size=TEST_SIZE,
    stratify=alllabs,
    random_state=RANDOM_STATE
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=VAL_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE
)

print(f" Train: {len(X_train_sub)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Convert to tensors
y_train_tens = torch.tensor(y_train_sub, dtype=torch.long)
y_val_tens = torch.tensor(y_val, dtype=torch.long)
y_test_tens = torch.tensor(y_test, dtype=torch.long)

/home/alant/repos/tal-projet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading DATASET movies
dataset size: 2000
 Train: 1280, Val: 320, Test: 400


In [2]:
def clear_gpu_memory_completely():
    """Aggressively clear ALL GPU memory between runs"""
    import gc
    import torch
    
    # 1. Delete ALL common GPU objects
    for name in dir():
        obj = globals().get(name)
        if isinstance(obj, torch.Tensor) and obj.is_cuda:
            del globals()[name]
        elif hasattr(obj, 'parameters') and hasattr(obj, 'to'):  # Models
            try:
                obj.cpu()
                del obj
            except:
                pass
    
    # 2. Force garbage collection
    gc.collect()
    
    # 3. Clear PyTorch cache (twice for stubborn memory)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.empty_cache()  # Double clear
    
    # 4. Reset CUDA context (nuclear option)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    
    print(f"GPU memory cleared. Allocated: {torch.cuda.memory_allocated()/1e6:.0f}MB, Cached: {torch.cuda.memory_reserved()/1e6:.0f}MB")

In [3]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

MAX_LENGTH = 512            
BATCH_SIZE = 8            
LEARNING_RATE = 2e-6
FREEZE_TRANSFORMER = False # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "FacebookAI/roberta-base"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), f"{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


<>:199: SyntaxWarning: invalid escape sequence '\S'
<>:199: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_8048/6847298.py:199: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


GPU memory cleared. Allocated: 0MB, Cached: 0MB
Using device: cuda
GPU memory available: 6.03 GB
Tokenizing train
Tokenizing validation
Tokenizing test

Model used (FacebookAI/roberta-base)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3826.42it/s]
RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training epochs :  20

Epoch 1/20
Model improved
train loss 0.6946 val loss 0.6922 | train f1 0.5818 val f1 0.6723 | train auc 0.4982 val auc 0.6144

Epoch 2/20
patience 1/4
train loss 0.6916 val loss 0.6875 | train f1 0.4955 val f1 0.5312 | train auc 0.5330 val auc 0.7177

Epoch 3/20
Model improved
train loss 0.6130 val loss 0.4470 | train f1 0.7099 val f1 0.8301 | train auc 0.8045 val auc 0.9068

Epoch 4/20
patience 1/4
train loss 0.3900 val loss 0.6254 | train f1 0.8493 val f1 0.8112 | train auc 0.8975 val auc 0.9075

Epoch 5/20
Model improved
train loss 0.3634 val loss 0.4063 | train f1 0.8669 val f1 0.8634 | train auc 0.9176 val auc 0.9318

Epoch 6/20
Model improved
train loss 0.3100 val loss 0.4050 | train f1 0.9027 val f1 0.8698 | train auc 0.9460 val auc 0.9431

Epoch 7/20
Model improved
train loss 0.2700 val loss 0.4093 | train f1 0.9197 val f1 0.8883 | train auc 0.9606 val auc 0.9491

Epoch 8/20
Model improved
train loss 0.2527 val loss 0.4061 | train f1 0.9260 val f1 0.8968 

In [4]:
clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()


GPU memory cleared. Allocated: 2012MB, Cached: 2122MB


254

In [ ]:
## w DROPOUT RoBERTa2

clear_gpu_memory_completely()
torch.cuda.empty_cache()
gc.collect()

MAX_LENGTH = 512            
BATCH_SIZE = 8            
LEARNING_RATE = 2e-6
FREEZE_TRANSFORMER = False # to freeze weights and only train classifier. 

# models : english: "bert-base-cased" "FacebookAI/roberta-base"
# french: "almanach/camemberta-base" "camembert-base"

# model selection
#MODEL_NAME = "almanach/camemberta-base"
MODEL_NAME = "FacebookAI/roberta-base"
SAVING_FILE_NAME = f"{DATASET}_transformer_{MAX_LENGTH}_{LEARNING_RATE}_{str(MODEL_NAME).replace('/', '_')}_regul_robertav3"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU memory available: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")

# tokenising
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a list of texts."""
    input_ids_list = []
    attention_masks_list = []
    
    for i, text in enumerate(texts):
        if (i + 1) % 2500 == 0:
            print(f"Tokens {i + 1}/{len(texts)}")
        
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors="pt",
            return_attention_mask=True
        )
        
        input_ids_list.append(encoded['input_ids'])
        attention_masks_list.append(encoded['attention_mask'])
    
    input_ids = torch.cat(input_ids_list, dim=0)
    attention_masks = torch.cat(attention_masks_list, dim=0)
    
    return input_ids, attention_masks

print("Tokenizing train")
input_ids_train, attention_masks_train = tokenize_dataset(X_train_sub, tokenizer)

print("Tokenizing validation")
input_ids_val, attention_masks_val = tokenize_dataset(X_val, tokenizer)

print("Tokenizing test")
input_ids_test, attention_masks_test = tokenize_dataset(X_test, tokenizer)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, y_train_tens)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, y_val_tens)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, y_test_tens)

train_dataloader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nModel used ({MODEL_NAME})")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

######
model.config.classifier_dropout = 0.2
####

if FREEZE_TRANSFORMER:
    base_model = getattr(model, model.config.model_type) 
    for param in base_model.parameters():
        param.requires_grad = False
        
    print(f"Base {model.config.model_type} frozen. Classifier head is still trainable.")

model = model.to(device)

#class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # weight_decay=0.01
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05) ################


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute evaluation metrics."""
    if DATASET == "pres":
        return {
            "f1": float(f1_score(all_labels, all_preds, average="macro")),
            "precision": float(precision_score(all_labels, all_preds, average="macro", zero_division=0)),
            "recall": float(recall_score(all_labels, all_preds, average="macro", zero_division=0)),
            "roc_auc": float(roc_auc_score(all_labels, all_probs)),
            "avg_precision": float(average_precision_score(all_labels, all_probs)),
        }
        
    return {
        "f1": float(f1_score(all_labels, all_preds, average="binary")),
        "precision": float(precision_score(all_labels, all_preds, average="binary", zero_division=0)),
        "recall": float(recall_score(all_labels, all_preds, average="binary", zero_division=0)),
        "roc_auc": float(roc_auc_score(all_labels, all_probs)),
        "avg_precision": float(average_precision_score(all_labels, all_probs)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
    }

def run_epoch(model, loader, optimizer, loss_fn, training=True):
    """Run one epoch of training or evaluation."""
    model.train() if training else model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for i, batch in enumerate(loader):
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

            if training:
                model.zero_grad()

            outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)

            loss_value = loss.item()
            total_loss += loss_value

            if training:
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    model.zero_grad()
                
                del scaled_loss, loss
            else:
                del loss

            preds = torch.argmax(logits, dim=1)
            
            probs = torch.softmax(logits.float(), dim=1)[:, 1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            del b_input_ids, b_attention_mask, b_labels, outputs, logits, preds, probs
            torch.cuda.empty_cache()
            gc.collect()

    all_probs = np.array(all_probs)

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = float(total_loss / len(loader))

    del all_probs, all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

history = []
best_f1 = 0.0
best_model_state = None
patience_counter = 0

print("Training epochs : ", EPOCHS)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_m = run_epoch(model, train_dataloader, optimizer, loss_fn, training=True)
    val_m = run_epoch(model, val_dataloader, optimizer, loss_fn, training=False)

    # for earl stopping with patience
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print("Model improved")
    else:
        patience_counter += 1
        print(f"patience {patience_counter}/{EARLY_STOPPING_PATIENCE}")

    history.append({"epoch": epoch + 1, "train": train_m, "val": val_m,})

    print(f"train loss {train_m['loss']:.4f} val loss {val_m['loss']:.4f} | "
            f"train f1 {train_m['f1']:.4f} val f1 {val_m['f1']:.4f} | "
            f"train auc {train_m['roc_auc']:.4f} val auc {val_m['roc_auc']:.4f}")

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\Stopping early at epoch {epoch+1}")
        break

print(f"\nBest validation F1 obtained: {best_f1:.4f}")

# doing for test
model.load_state_dict(best_model_state)
torch.save(model.state_dict(), f"{SAVING_FILE_NAME}.pth")

test_m = run_epoch(model, test_dataloader, optimizer, loss_fn, training=False)

print(f"\nTest | loss {test_m['loss']:.4f} | f1 {test_m['f1']:.4f} | "
    f"prec {test_m['precision']:.4f} | rec {test_m['recall']:.4f} | "
    f"auc {test_m['roc_auc']:.4f} | ap {test_m['avg_precision']:.4f}")

results = {
    "run_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "frozen": FREEZE_TRANSFORMER,
    },
    "best_val_f1": best_f1,
    "epochs_trained": len(history),
    "history": history,
    "test": test_m,
}

with open(f"{SAVING_FILE_NAME}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {SAVING_FILE_NAME}_results.json")
print(f"Model saved to {SAVING_FILE_NAME}.pth")


<>:205: SyntaxWarning: invalid escape sequence '\S'
<>:205: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_33322/3959915086.py:205: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Stopping early at epoch {epoch+1}")


GPU memory cleared. Allocated: 0MB, Cached: 0MB
Using device: cuda
GPU memory available: 6.03 GB
Tokenizing train
Tokenizing validation
Tokenizing test

Model used (FacebookAI/roberta-base)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6935.06it/s]
RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training epochs :  20

Epoch 1/20
